In [ ]:
# install color extraction library
!pip install colorgram.py

In [ ]:
import colorgram
import json
import requests
import webcolors

from io import BytesIO
from os import makedirs
from PIL import Image as PImage
from time import sleep

# make an empty directory to download images into
makedirs("images/500", exist_ok=True)

GQL_URL = "https://api.cooperhewitt.org/"

In [ ]:
# number of objects to get per query
OBJS_PER_QUERY = 250

# term to search for
QUERY_TERM = "flag"
# QUERY_TERM = "poster"

# GraphQL search string. This is messy.
graphqlQuery = ""
graphqlQuery += "{"
graphqlQuery += f"object(general: \"{QUERY_TERM}\", hasImages:true, size: 1, page: 1, sort:[{{id: \"asc\"}}]) {{"
graphqlQuery += "id"
graphqlQuery += "}"
graphqlQuery += "}"

# get a response and decode json into variable
response = requests.get(f"{GQL_URL}?query={graphqlQuery}")
data = response.json()

# total number of objects
n_rows = data["extensions"]["pagination"]["hits"]

# wait 1 second to make a new request or else Cooper Hewitt gets angry
sleep(1)

# empty list to store objects
objects = []

# loop to get objects 250 rows at a time
for qcnt in range(n_rows // OBJS_PER_QUERY + 1):
  # build the full query string again. Still messy.
  graphqlQuery = ""
  graphqlQuery += "{"
  graphqlQuery += f"object(general: \"{QUERY_TERM}\", hasImages:true, size: {OBJS_PER_QUERY}, page: {qcnt}, sort:[{{id: \"asc\"}}]) {{"
  graphqlQuery += """
  id
  title
  summary
  date
  description
  color
  multimedia
  geography"""
  graphqlQuery += "}"
  graphqlQuery += "}"

  # get a response and decode json into variable
  response = requests.get(f"{GQL_URL}?query={graphqlQuery}")
  data = response.json()

  # iterate over returned objects/rows
  for row in data["data"]["object"]:
    # counter for current object index
    ocnt = len(objects)

    # print progress and save json every 25 objects
    if ocnt % 25 == 0:
      print(ocnt, "/", n_rows)
      with open("data.json", "w") as ofp:
        json.dump(objects, ofp)

    # try to extract image info from nested object
    try:
      # if some of the properties are missing, this will throw an error
      row_media = row["content"]["descriptiveNonRepeating"]["online_media"]["media"][0]
    except:
      # and we assume the image info is missing
      row_media = {}

    # try to get the url for the first image of this object
    try:
      mediaUrl = row["multimedia"][0]["preview"]["url"]
    except:
      # if it doesn't exist set it to None
      mediaUrl = None

    # try to get the media type for the first image of this object
    try:
      mediaType = row["multimedia"][0]["type"]
    except:
      # if it doesn't exist set it to None
      mediaType = None

    # if url exists and it's for an image
    if mediaUrl and mediaType == "image":
      # try to download the image
      try:
        # download image, resize it and save it
        response = requests.get(mediaUrl)
        response.raise_for_status()
        img = PImage.open(BytesIO(response.content)).convert("RGB")
        img.thumbnail((500, 500))
        img.save(f"images/500/{row['id']}.jpg")

        # extract the 8 dominant colors from the image
        colors = colorgram.extract(img, 8)

        # save color info (proportions, hex values and rgb values)
        row["colors"] = {
          "proportions": [round(c.proportion, 3) for c in colors],
          "hex": [webcolors.rgb_to_hex(c.rgb) for c in colors],
          "rgb": [tuple(c.rgb) for c in colors],
        }

        # also save url at the root of the object
        row["image_url"] = mediaUrl

        # add to list that gets saved as json
        objects.append(row)

      # if we can't download the image, skip this row
      except requests.exceptions.HTTPError as errh:
        pass

    # slow down to prevent getting kicked out of Cooper Hewitt's API
    sleep(0.05)
  sleep(1)

print(len(objects))

In [ ]:
# save final version of the json
with open("data.json", "w") as ofp:
  json.dump(objects, ofp)

In [ ]:
# compresses the directory with all of the images
!tar -czf images.tgz images/